# 2026 최종 제출용 추론 노트북 — Corrected Exact-Lag Ensemble

**이 노트북을 Save & Run All 한 버전을 제출합니다.**

최종 모델은 **2019~2025 전체 학습**이며, LSTM은 사용하지 않습니다.

- 입력: 매일 **14:00 KST GK-2A LE1B/KO 16채널**
- 추가 입력: 정확히 **1일 전 / 2일 전 14:00 위성값**
- 피처: spatial nearest-3, daily anomaly, exact-date lag/delta/rolling
- 결측 lag를 `bfill`하지 않음
- TA 기본 앙상블: **0.40 LightGBM + 0.60 CatBoost + 0.00 XGBoost**
- HM 기본 앙상블: **0.30 LightGBM + 0.40 CatBoost + 0.30 XGBoost**
- LSTM: **OFF**

> 2026년 7월 pseudo-validation에서 다른 가중치를 선택했다면 모델 재학습 없이 번들의 `manifest.json` 가중치만 교체할 수 있습니다.

---

## 지켜야 할 규칙

1. 운영진은 `API_KEY`, `PRED_START`, `PRED_END`만 바꿉니다.
2. 예측 대상 시각은 매일 **14:00 KST**입니다.
3. 평가기간 ASOS TA/HM은 사용하지 않습니다.
4. 평가일 위성 및 평가일 이전 위성만 사용합니다.
5. 마지막에 `pred`와 `/kaggle/working/submission.csv`가 생성되어야 합니다.

## Add Input

- 공식 대회 Dataset (`station_list.csv`)
- 모델 Dataset: `sme_corrected_exactlag_ensemble_2026_v2.zip`

**Session options → Internet: On**


## 1. 설정 — 수정하지 마세요

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  ★ 운영진 수정 구역 — 채점 시 아래 3줄만 교체합니다 ★               ║
# ╚═══════════════════════════════════════════════════════════════════╝
API_KEY    = ""                # 기상청 API Hub 인증키
PRED_START = "20260824"        # 예측 시작일 (YYYYMMDD)
PRED_END   = "20260830"        # 예측 종료일 (YYYYMMDD)
# ═══════════════════════════════════════════════════════════════════════

import os, sys, glob
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests

if not API_KEY:
    sys.exit("API_KEY를 입력하세요. (Kaggle Secrets는 사용할 수 없습니다)")

# ── 평가 기준 시각: 수정하지 마세요 ─────────────────────────────
EVAL_HOUR = 14
EVAL_MINUTE = 0

# 예측 대상 일시
# 중요: PRED_DATES의 각 원소는 '날짜만'이 아니라 매일 14:00 KST의 datetime 입니다.
_start_dt = datetime.strptime(PRED_START, "%Y%m%d").replace(
    hour=EVAL_HOUR, minute=EVAL_MINUTE, second=0, microsecond=0
)
_last_dt = datetime.strptime(PRED_END, "%Y%m%d").replace(
    hour=EVAL_HOUR, minute=EVAL_MINUTE, second=0, microsecond=0
)

if _start_dt > _last_dt:
    sys.exit("PRED_START는 PRED_END보다 늦을 수 없습니다.")

PRED_DATES = []
_dt = _start_dt
while _dt <= _last_dt:
    PRED_DATES.append(_dt)
    _dt += timedelta(days=1)

# 방어적 검증: 예측 대상 시각이 실수로 00시 등으로 바뀌는 것을 차단
if any(
    (pd.Timestamp(d).hour != EVAL_HOUR) or
    (pd.Timestamp(d).minute != EVAL_MINUTE)
    for d in PRED_DATES
):
    sys.exit("PRED_DATES 생성 오류: 모든 예측 대상 시각은 14:00 KST여야 합니다.")

def _as_kst(dt):
    """naive datetime은 KST로 해석하고, timezone-aware 값은 KST로 변환합니다."""
    ts = pd.Timestamp(dt)
    if ts.tzinfo is None:
        return ts.tz_localize("Asia/Seoul")
    return ts.tz_convert("Asia/Seoul")

# 위성 API 요청용 시각 문자열 생성 함수
def to_api_datetime(obs_dt, target_dt=None):

    if target_dt is None:
        target_dt = obs_dt

    obs = _as_kst(obs_dt)
    target = _as_kst(target_dt)

    
    if target.hour != EVAL_HOUR or target.minute != EVAL_MINUTE:
        raise ValueError(
            f"잘못된 예측 대상 시각: {target}. "
            f"target_dt는 {EVAL_HOUR:02d}:{EVAL_MINUTE:02d} KST여야 합니다."
        )

   
    if obs > target:
        raise ValueError(
            f"미래 위성영상 사용 금지: obs_dt={obs}, target_dt={target}. "
            "추론에는 예측 대상 시각까지 관측된 위성영상만 사용할 수 있습니다."
        )

    # KMA GK-2A LE1B API date는 UTC 기준
    return obs.tz_convert("UTC").strftime("%Y%m%d%H%M")


from urllib.parse import urlparse, parse_qs

API_AUDIT_LOG = []

if not hasattr(requests.sessions.Session, "_competition_original_request"):
    requests.sessions.Session._competition_original_request = requests.sessions.Session.request

_ORIGINAL_REQUEST = requests.sessions.Session._competition_original_request

def _audit_request(self, method, url, **kwargs):
    try:
        parsed = urlparse(str(url))
        host = parsed.netloc

        if "apihub.kma.go.kr" in host:
            query = parse_qs(parsed.query, keep_blank_values=True)

            params = kwargs.get("params")
            if isinstance(params, dict):
                for k, v in params.items():
                    if k == "authKey":
                        continue
                    if isinstance(v, (list, tuple)):
                        query[k] = [str(x) for x in v]
                    else:
                        query[k] = [str(v)]

            def _first(name):
                vals = query.get(name, [])
                return vals[0] if vals else None

            API_AUDIT_LOG.append({
                "method": str(method).upper(),
                "host": host,
                "path": parsed.path,
                "date": _first("date"),
                "sDate": _first("sDate"),
                "eDate": _first("eDate"),
            })
    except Exception as e:
        print(f"[경고] API 로그 기록 실패: {type(e).__name__}: {e}")

    return _ORIGINAL_REQUEST(self, method, url, **kwargs)

requests.sessions.Session.request = _audit_request

# 평가 대상 지점 ── station_list.csv 에 정의된 공식 96개
# glob 반환 순서에 의존하지 않고, STN_ID 96개 후보를 검증해서 선택합니다.
_hits = sorted(glob.glob("/kaggle/input/**/station_list.csv", recursive=True))
if not _hits:
    sys.exit("station_list.csv 를 찾을 수 없습니다. Add Input을 확인하세요.")

_valid_station_files = []
for _p in _hits:
    try:
        _tmp = pd.read_csv(_p)
        if "STN_ID" not in _tmp.columns:
            continue
        _ids = tuple(sorted(_tmp["STN_ID"].dropna().astype(int).unique().tolist()))
        if len(_ids) == 96:
            _valid_station_files.append((_p, _ids))
    except Exception:
        continue

if not _valid_station_files:
    sys.exit(
        "STN_ID 96개를 가진 station_list.csv를 찾지 못했습니다. "
        "공식 대회 데이터셋 연결을 확인하세요."
    )

_station_id_sets = {ids for _, ids in _valid_station_files}
if len(_station_id_sets) > 1:
    _paths = [p for p, _ in _valid_station_files]
    sys.exit(
        "서로 다른 96지점 station_list.csv가 여러 개 발견되었습니다. "
        "공식 대회 데이터셋만 남기거나 중복 파일명을 변경하세요.\n"
        + "\n".join(_paths)
    )

_station_path, _station_ids = _valid_station_files[0]
STATIONS = list(_station_ids)

if len(_valid_station_files) > 1:
    print(
        f"[경고] 동일한 96지점 station_list.csv가 {len(_valid_station_files)}개 발견되었습니다. "
        f"다음 파일을 사용합니다: {_station_path}"
    )
else:
    print(f"station_list.csv: {_station_path}")

print(f"예측 기간 : {PRED_START} ~ {PRED_END}  ({len(PRED_DATES)}일)")
print(f"예측 대상 시각 : 매일 {EVAL_HOUR:02d}:{EVAL_MINUTE:02d} KST")
print(f"평가 지점 : {len(STATIONS)}개")
print(
    "API 시각 예시 : "
    f"{PRED_DATES[0].strftime('%Y-%m-%d %H:%M')} KST -> "
    f"{to_api_datetime(PRED_DATES[0])} UTC"
)

## 2. 자유 구현 — Corrected Exact-Lag Ensemble

이 셀은 학습된 번들을 불러와 **추론만** 합니다.

각 평가일 `D`에 대해 `D-2`, `D-1`, `D`의 **14:00 KST** 위성 16채널을 사용합니다.  
따라서 7일 평가라면 총 9일 × 16채널의 위성 파일이 필요합니다.

피처 엔지니어링은 학습과 동일하게 수행되며 `groupby().shift()`나 `bfill()`을 쓰지 않고 **정확한 달력 날짜 차이**로 lag를 생성합니다.

### `pred` 반환 형식

| 컬럼 | 내용 |
|---|---|
| `Date` | 정수 `YYYYMMDD` |
| `STN_ID` | 정수 지점번호 |
| `TA` | 기온 예측값 |
| `HM` | 습도 예측값 |


In [ ]:
# 추론 환경 + Corrected Exact-Lag 모델 번들 로드
import importlib.util
import json
import subprocess
import zipfile

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "lightgbm==4.6.0",
        "xgboost==3.1.3",
        "catboost==1.2.8",
        "pyproj==3.7.2",
        "xarray",
        "h5netcdf",
        "scipy",
        "joblib",
    ],
    check=True,
)

BUNDLE_BASENAME = "sme_corrected_exactlag_ensemble_2026_v2.zip"

# 1) 압축 해제된 manifest 탐색
_bundle_candidates = []
for _manifest in glob.glob("/kaggle/input/**/manifest.json", recursive=True):
    try:
        with open(_manifest, encoding="utf-8") as _f:
            _meta = json.load(_f)
        if str(_meta.get("bundle_version", "")).startswith("2026.corrected_exactlag"):
            _bundle_candidates.append(os.path.dirname(_manifest))
    except Exception:
        pass

# 2) ZIP 한 개만 Add Input 한 경우 자동 압축 해제
if not _bundle_candidates:
    _zips = sorted(glob.glob(f"/kaggle/input/**/{BUNDLE_BASENAME}", recursive=True))
    if len(_zips) != 1:
        raise RuntimeError(
            f"{BUNDLE_BASENAME}을 정확히 하나 연결해야 합니다. 발견={_zips}"
        )
    _extract_dir = "/kaggle/working/corrected_exactlag_bundle"
    os.makedirs(_extract_dir, exist_ok=True)
    with zipfile.ZipFile(_zips[0]) as _z:
        _z.extractall(_extract_dir)
    _bundle_candidates = [_extract_dir]

if len(_bundle_candidates) != 1:
    raise RuntimeError(f"모델 번들 후보가 정확히 하나가 아닙니다: {_bundle_candidates}")

BUNDLE_DIR = _bundle_candidates[0]
with open(os.path.join(BUNDLE_DIR, "manifest.json"), encoding="utf-8") as _f:
    MANIFEST = json.load(_f)

print("bundle_version:", MANIFEST["bundle_version"])
print("training_date_range:", MANIFEST["training_date_range"])
print("TA weights:", MANIFEST["weights"]["TA"])
print("HM weights:", MANIFEST["weights"]["HM"])
print("LSTM enabled:", MANIFEST["lstm_enabled"])

if MANIFEST.get("lstm_enabled", True):
    raise RuntimeError("이 제출본은 LSTM OFF 번들이어야 합니다.")

_module_path = os.path.join(BUNDLE_DIR, "submission_2026_inference.py")
_spec = importlib.util.spec_from_file_location("submission_2026_inference", _module_path)
_inference = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_inference)

_station_frame = pd.read_csv(_station_path)

# D-2, D-1, D 매일 14:00 KST 위성값으로 exact lag 피처 생성 후 예측
pred = _inference.predict_from_api(
    api_key=API_KEY,
    pred_dates=PRED_DATES,
    stations=_station_frame,
    bundle_dir=BUNDLE_DIR,
    to_api_datetime=to_api_datetime,
    cache_dir="/kaggle/working/gk2a_exactlag_cache",
    failure_csv="/kaggle/working/gk2a_exactlag_failures.csv",
    max_missing_fraction=0.0,
    timeout_seconds=90.0,
    max_retries=4,
    request_interval_seconds=0.15,
)

pred = pred[["Date", "STN_ID", "TA", "HM"]].copy()
print(pred.describe(include="all").to_string())

## 3. 제출 파일 생성 — 수정하지 마세요

In [ ]:
# ── PRED_DATES 표현 점검 ─────────────────────────────────────
# 참가자가 자유 구현 중 PRED_DATES를 date/UTC Timestamp 등으로 바꿔도
# 최종 제출 행 검증은 PRED_START/PRED_END에서 독립적으로 수행합니다.
# 따라서 여기서는 중단하지 않고 경고만 합니다.
_bad_times = []
for _d in PRED_DATES:
    try:
        _ts = pd.Timestamp(_d)
        if _ts.tzinfo is None:
            _kst = _ts
        else:
            _kst = _ts.tz_convert("Asia/Seoul").tz_localize(None)

        if _kst.hour != EVAL_HOUR or _kst.minute != EVAL_MINUTE:
            _bad_times.append(_d)
    except Exception:
        _bad_times.append(_d)

if _bad_times:
    print(
        "[경고] 현재 PRED_DATES에 14:00 KST가 아닌 표현이 포함되어 있습니다. "
        "최종 제출 날짜/행 수는 PRED_START/PRED_END에서 독립 검증합니다. "
        f"예시: {_bad_times[:3]}"
    )

# ── KMA API 감사 로그 저장 ───────────────────────────────────
_audit_out = "/kaggle/working/api_audit_log.csv"
if "API_AUDIT_LOG" in globals():
    _audit_df = pd.DataFrame(
        API_AUDIT_LOG,
        columns=["method", "host", "path", "date", "sDate", "eDate"],
    )
    _audit_df.to_csv(_audit_out, index=False)

    if len(_audit_df):
        _non_le1b = _audit_df[
            ~_audit_df["path"].fillna("").str.contains("/LE1B/", regex=False)
        ]
        if len(_non_le1b):
            print(
                f"[경고] /LE1B/ 이외 KMA API 경로 호출이 {len(_non_le1b)}건 기록되었습니다. "
                f"감사 로그: {_audit_out}"
            )
        else:
            print(f"API 감사 로그 저장: {_audit_out} ({len(_audit_df)}건)")
    else:
        print(f"API 감사 로그 저장: {_audit_out} (기록 0건)")

# ── pred 검증 ─────────────────────────────────────────────
if "pred" not in dir():
    sys.exit("자유 구현 영역에서 'pred' DataFrame을 만들어야 합니다.")
if not isinstance(pred, pd.DataFrame):
    sys.exit(f"pred 는 DataFrame 이어야 합니다 (현재: {type(pred).__name__})")

need = {"Date", "STN_ID", "TA", "HM"}
if not need <= set(pred.columns):
    sys.exit(f"pred 컬럼 부족: {sorted(need - set(pred.columns))}")

# 날짜 × 지점의 모든 조합이 정확히 한 번씩 있어야 합니다
# 기대 날짜 × 지점 조합은 자유 구현 영역에서 바뀔 수 있는 PRED_DATES가 아니라
# 운영진 설정값 PRED_START/PRED_END에서 독립적으로 재계산합니다.
_ref_dates = []
_ref_d = datetime.strptime(PRED_START, "%Y%m%d")
_ref_end = datetime.strptime(PRED_END, "%Y%m%d")
while _ref_d <= _ref_end:
    _ref_dates.append(_ref_d)
    _ref_d += timedelta(days=1)

_want = {
    (int(d.strftime("%Y%m%d")), s)
    for d in _ref_dates
    for s in STATIONS
}
_got  = [(int(a), int(b)) for a, b in
         zip(pred["Date"].astype(int), pred["STN_ID"].astype(int))]
if len(_got) != len(set(_got)):
    sys.exit("pred 에 중복된 (Date, STN_ID) 조합이 있습니다.")
if set(_got) != _want:
    miss, extra = _want - set(_got), set(_got) - _want
    sys.exit(f"pred 행 구성 오류 — 누락 {len(miss)}개, 불필요 {len(extra)}개\n"
             f"  누락 예시: {sorted(miss)[:3]}\n"
             f"  불필요 예시: {sorted(extra)[:3]}")

# ── 제출 파일 구성 ────────────────────────────────────────
submission = pd.DataFrame({
    "ID": pred["Date"].astype(int).astype(str) + "_"
          + pred["STN_ID"].astype(int).astype(str),
    "TA": np.clip(pd.to_numeric(pred["TA"], errors="coerce"), -50, 50).round(2),
    "HM": np.clip(pd.to_numeric(pred["HM"], errors="coerce"), 0, 100).round(2),
}).sort_values("ID").reset_index(drop=True)

if submission[["TA", "HM"]].isna().any().any():
    n = int(submission[["TA", "HM"]].isna().any(axis=1).sum())
    sys.exit(f"결측 예측값 {n}행 — 모든 행에 값이 있어야 합니다.")

out = "/kaggle/working/submission.csv"
submission.to_csv(out, index=False)

print("=" * 52)
print(f"  저장 완료: {out}")
print(f"  {len(submission)}행 = {len(STATIONS)}지점 x {len(_ref_dates)}일")
print(f"  TA {submission.TA.min():.1f} ~ {submission.TA.max():.1f} C")
print(f"  HM {submission.HM.min():.1f} ~ {submission.HM.max():.1f} %")
print("=" * 52)
print(submission.head().to_string(index=False))

## 제출 전 점검

- [ ] `PRED_START` / `PRED_END` 외에 평가 날짜가 하드코딩되어 있지 않은가
- [ ] 모델 번들 버전이 `2026.corrected_exactlag...` 인가
- [ ] `training_date_range`가 2019~2025인가
- [ ] `LSTM enabled: False`가 출력되는가
- [ ] 평가일마다 14:00 KST 위성값을 사용하는가
- [ ] exact lag를 위해 D-1 / D-2 14:00 KST만 추가로 사용하는가
- [ ] 평가기간 ASOS TA/HM을 호출하거나 입력하지 않는가
- [ ] `/GK2A/LE1B/.../KO/data` 외 KMA API를 사용하지 않는가
- [ ] `submission.csv`가 날짜수 × 96행이며 TA/HM 결측이 없는가
- [ ] 모델 Dataset이 운영진이 접근 가능하도록 공유되어 있는가

### 최종 평가 기본값

`2026-08-24 ~ 2026-08-30` / 매일 `14:00 KST`
